# rizzvision-v2: EfficientNetB3 T-Shirt Classifier
# 
# Setup: attach nitin2807/rizzvision-tshirt-dataset as input, enable GPU T4 x1, run all cells.
# Outputs: tshirt_classifier.keras + threshold.json (download from Output tab)

In [ ]:
import os
import json
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.metrics import roc_auc_score, confusion_matrix, roc_curve

print('TF version:', tf.__version__)
print('GPU:', tf.config.list_physical_devices('GPU'))

OUT_DIR = '/kaggle/working'
DATASET_DIR = '/kaggle/input/datasets/nitin2807/rizzvision-tshirt-dataset'
print('Dataset dir:', DATASET_DIR)
print('Contents:', os.listdir(DATASET_DIR))

INPUT_SIZE = 300
BATCH_SIZE = 32
AUTOTUNE = tf.data.AUTOTUNE

In [ ]:
def augment(image, label):
    image = tf.image.random_flip_left_right(image)
    image = tf.image.random_brightness(image, 50.0)
    image = tf.image.random_contrast(image, 0.8, 1.2)
    image = tf.image.random_saturation(image, 0.8, 1.2)
    crop_size = tf.random.uniform([], minval=int(INPUT_SIZE * 0.85), maxval=INPUT_SIZE, dtype=tf.int32)
    image = tf.image.random_crop(image, size=[crop_size, crop_size, 3])
    image = tf.image.resize(image, [INPUT_SIZE, INPUT_SIZE])
    image = tf.clip_by_value(image, 0.0, 255.0)
    return image, label

def load_dataset(split, augment_data=False):
    ds = keras.utils.image_dataset_from_directory(
        os.path.join(DATASET_DIR, split),
        image_size=(INPUT_SIZE, INPUT_SIZE),
        batch_size=None,
        label_mode='binary',
        shuffle=(split == 'train'),
        seed=42,
    )
    # EfficientNetB3 expects [0, 255] — cast only, do not normalize
    ds = ds.map(lambda x, y: (tf.cast(x, tf.float32), y), num_parallel_calls=AUTOTUNE)
    if augment_data:
        ds = ds.map(augment, num_parallel_calls=AUTOTUNE)
    ds = ds.batch(BATCH_SIZE)
    return ds.prefetch(AUTOTUNE)

train_ds = load_dataset('train', augment_data=True)
val_ds = load_dataset('val')
test_ds = load_dataset('test')
print('Datasets loaded')

In [ ]:
def build_model(trainable_base=False):
    base = keras.applications.EfficientNetB3(
        include_top=False,
        weights='imagenet',
        input_shape=(INPUT_SIZE, INPUT_SIZE, 3),
    )
    base.trainable = trainable_base
    inputs = keras.Input(shape=(INPUT_SIZE, INPUT_SIZE, 3))
    x = base(inputs, training=False)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(0.4)(x)
    x = layers.Dense(256, activation='relu')(x)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(0.3)(x)
    outputs = layers.Dense(1, activation='sigmoid')(x)
    return keras.Model(inputs, outputs), base

model, base = build_model(trainable_base=False)
model.summary()

In [ ]:
model.compile(
    optimizer=keras.optimizers.Adam(1e-3),
    loss=keras.losses.BinaryFocalCrossentropy(gamma=2.0),
    metrics=['accuracy', keras.metrics.AUC(name='auc')],
)

phase1 = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=5,
    callbacks=[
        keras.callbacks.ModelCheckpoint(
            f'{OUT_DIR}/best_phase1.keras', save_best_only=True, monitor='val_auc', mode='max'
        ),
    ],
)
print('Phase 1 complete')

In [ ]:
base.trainable = True
for layer in base.layers[:-50]:
    layer.trainable = False

lr_schedule = keras.optimizers.schedules.CosineDecay(
    initial_learning_rate=1e-4,
    decay_steps=25 * len(train_ds),
    alpha=1e-6,
)

model.compile(
    optimizer=keras.optimizers.Adam(lr_schedule),
    loss=keras.losses.BinaryFocalCrossentropy(gamma=2.0),
    metrics=['accuracy', keras.metrics.AUC(name='auc')],
)

phase2 = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=25,
    callbacks=[
        keras.callbacks.ModelCheckpoint(
            f'{OUT_DIR}/tshirt_classifier.keras', save_best_only=True, monitor='val_auc', mode='max'
        ),
        keras.callbacks.EarlyStopping(monitor='val_auc', mode='max', patience=7, restore_best_weights=True),
        keras.callbacks.ReduceLROnPlateau(monitor='val_auc', mode='max', factor=0.5, patience=3, verbose=1),
    ],
)
print('Phase 2 complete')

In [ ]:
best_model = keras.models.load_model(f'{OUT_DIR}/tshirt_classifier.keras')

val_probs, val_labels = [], []
for batch_x, batch_y in val_ds:
    val_probs.extend(best_model.predict(batch_x, verbose=0).flatten().tolist())
    val_labels.extend(batch_y.numpy().flatten().tolist())

val_probs = np.array(val_probs)
val_labels = np.array(val_labels)
print(f'Val AUC: {roc_auc_score(val_labels, val_probs):.4f}')

fpr_arr, tpr_arr, thresholds = roc_curve(val_labels, val_probs)
best_thresh, best_f1 = 0.87, 0.0
for t, fp, tp in zip(thresholds, fpr_arr, tpr_arr):
    if fp > 0.02:
        continue
    precision = tp / (tp + (1 - tp) + 1e-9)
    recall = tp
    f1 = 2 * precision * recall / (precision + recall + 1e-9)
    if f1 > best_f1:
        best_f1, best_thresh = f1, float(t)

print(f'Optimal threshold (FPR <= 2%): {best_thresh:.4f}')
with open(f'{OUT_DIR}/threshold.json', 'w') as f:
    json.dump({'threshold': round(best_thresh, 4)}, f, indent=2)

In [ ]:
test_probs, test_labels = [], []
for batch_x, batch_y in test_ds:
    test_probs.extend(best_model.predict(batch_x, verbose=0).flatten().tolist())
    test_labels.extend(batch_y.numpy().flatten().tolist())

test_probs = np.array(test_probs)
test_labels = np.array(test_labels)
test_preds = (test_probs >= best_thresh).astype(int)

cm = confusion_matrix(test_labels, test_preds)
tn, fp, fn, tp = cm.ravel()
accuracy = (tp + tn) / len(test_labels)
fpr = fp / (fp + tn)
fnr = fn / (fn + tp)

print('=== Test Set Results ===')
print(f'Accuracy: {accuracy:.4f}  (target >= 0.93)')
print(f'AUC-ROC:  {roc_auc_score(test_labels, test_probs):.4f}  (target >= 0.97)')
print(f'FPR:      {fpr:.4f}  (target <= 0.02)')
print(f'FNR:      {fnr:.4f}  (target <= 0.08)')
print(f'Threshold used: {best_thresh}')
print('\nConfusion matrix:')
print(cm)

if accuracy >= 0.93 and fpr <= 0.02:
    print('\n✓ All targets met — model ready for deployment')
else:
    print('\n✗ Targets not met')

print(f'\nDownload tshirt_classifier.keras and threshold.json from the Output tab')